In [4]:
import pandas as pd
import sqlite3
import kagglehub
import os

# --- PASO 1: DESCARGA ---
path = kagglehub.dataset_download("grandmaster07/student-exam-performance-dataset-analysis")
archivos = os.listdir(path)
csv_file = [f for f in archivos if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f"¡Dataset '{csv_file}' cargado con éxito!")

# --- PASO 2: BASE DE DATOS ---
conn = sqlite3.connect('escuela.db')
cursor = conn.cursor()

cursor.executescript('''
DROP TABLE IF EXISTS Calificaciones;
DROP TABLE IF EXISTS Entorno;
DROP TABLE IF EXISTS Estudiantes;

CREATE TABLE Estudiantes (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    gender TEXT
);

CREATE TABLE Entorno (
    env_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    parent_education TEXT,
    family_income TEXT,
    access_to_resources TEXT,
    FOREIGN KEY (student_id) REFERENCES Estudiantes(student_id)
);

CREATE TABLE Calificaciones (
    exam_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    hours_studied INTEGER,
    attendance INTEGER,
    exam_score INTEGER,
    FOREIGN KEY (student_id) REFERENCES Estudiantes(student_id)
);
''')

# --- PASO 3: INSERCIÓN DE DATOS ---
# 1. Estudiantes
estudiantes_df = df[['Gender']].copy()
estudiantes_df.columns = ['gender']
estudiantes_df.to_sql('Estudiantes', conn, if_exists='append', index=False)

# Obtener IDs
df_ids = pd.read_sql("SELECT student_id FROM Estudiantes", conn)
df['student_id'] = df_ids['student_id']

# 2. Entorno
entorno_df = df[['student_id', 'Parental_Education_Level', 'Family_Income', 'Access_to_Resources']].copy()
entorno_df.columns = ['student_id', 'parent_education', 'family_income', 'access_to_resources']
entorno_df.to_sql('Entorno', conn, if_exists='append', index=False)

# 3. Calificaciones
calificaciones_df = df[['student_id', 'Hours_Studied', 'Attendance', 'Exam_Score']].copy()
calificaciones_df.columns = ['student_id', 'hours_studied', 'attendance', 'exam_score']
calificaciones_df.to_sql('Calificaciones', conn, if_exists='append', index=False)

conn.commit()
print("Base de datos 'escuela.db' creada correctamente con el nuevo dataset.")

# --- PASO 4: PRUEBA DE CONSULTA ---
print("\n--- Vista previa de los datos insertados ---")
print(pd.read_sql("SELECT * FROM Calificaciones LIMIT 5", conn))
conn.close()

Using Colab cache for faster access to the 'student-exam-performance-dataset-analysis' dataset.
¡Dataset 'StudentPerformanceFactors.csv' cargado con éxito!
Base de datos 'escuela.db' creada correctamente con el nuevo dataset.

--- Vista previa de los datos insertados ---
   exam_id  student_id  hours_studied  attendance  exam_score
0        1           1             23          84          67
1        2           2             19          64          61
2        3           3             24          98          74
3        4           4             29          89          71
4        5           5             19          92          70


In [6]:
conn = sqlite3.connect('escuela.db')
# --- CONSULTA 1: Extracción y Transformación ---
# Objetivo: Promedio de notas por Género
query1 = """
SELECT gender as Genero,
       ROUND(AVG(exam_score), 2) as Promedio_Examen
FROM Estudiantes
JOIN Calificaciones ON Estudiantes.student_id = Calificaciones.student_id
GROUP BY gender;
"""
print("1. Promedio por Género:")
display(pd.read_sql(query1, conn))

# --- CONSULTA 2: Refinamiento y Control ---
# Objetivo: Estudiantes con alto puntaje pero pocos recursos
query2 = """
SELECT student_id, exam_score
FROM Calificaciones
WHERE exam_score > 90
AND student_id IN (SELECT student_id FROM Entorno WHERE access_to_resources = 'Low')
ORDER BY exam_score DESC;
"""
print("\n2. Estudiantes destacados con pocos recursos:")
display(pd.read_sql(query2, conn))

# --- CONSULTA 3: Agrupación ---
# Objetivo: Relación entre Ingresos Familiares y asistencia promedio
query3 = """
SELECT family_income,
       COUNT(*) as Total_Estudiantes,
       AVG(attendance) as Asistencia_Media
FROM Entorno
JOIN Calificaciones ON Entorno.student_id = Calificaciones.student_id
GROUP BY family_income;
"""
print("\n3. Análisis por Ingresos Familiares:")
display(pd.read_sql(query3, conn))

# --- CONSULTA 4: Filtrado Avanzado ---
# Objetivo: Impacto de las horas de estudio en el éxito (Score > 80)
query4 = """
SELECT parent_education,
       AVG(hours_studied) as Horas_Promedio
FROM Entorno
JOIN Calificaciones ON Entorno.student_id = Calificaciones.student_id
WHERE exam_score > 80
GROUP BY parent_education;
"""
print("\n4. Horas de estudio de alumnos sobresalientes según educación parental:")
display(pd.read_sql(query4, conn))

# --- CONSULTA 5: Resumen General ---
# Objetivo: Top 5 de estudiantes con mejor balance estudio/nota
query5 = """
SELECT student_id,
       hours_studied,
       exam_score
FROM Calificaciones
WHERE attendance > 90
ORDER BY exam_score DESC
LIMIT 5;
"""
print("\n5. Top 5 estudiantes con asistencia perfecta:")
display(pd.read_sql(query5, conn))

conn.close()

1. Promedio por Género:


,Genero,Promedio_Examen
0,Female,67.24
1,Male,67.23



2. Estudiantes destacados con pocos recursos:


,student_id,exam_score
0,6348,98



3. Análisis por Ingresos Familiares:


,family_income,Total_Estudiantes,Asistencia_Media
0,High,1269,79.907801
1,Low,2672,80.215195
2,Medium,2666,79.772318



4. Horas de estudio de alumnos sobresalientes según educación parental:


,parent_education,Horas_Promedio
0,College,19.214286
1,High School,19.045455
2,Postgraduate,22.428571



5. Top 5 estudiantes con asistencia perfecta:


,student_id,hours_studied,exam_score
0,1526,27,101
1,6348,28,98
2,5967,25,97
3,3458,18,96
4,771,24,94


In [16]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

st.set_page_config(page_title="Dashboard Académico", layout="wide")

st.title("📊 Dashboard de Desempeño Estudiantil")
st.markdown("Análisis basado en factores socioeconómicos y académicos.")

conn = sqlite3.connect('escuela.db')

# --- SIDEBAR ---
st.sidebar.header("Filtros de Análisis")
nivel_educativo = st.sidebar.multiselect(
    "Nivel Educativo de Padres:",
    options=["High School", "Associate's Degree", "Bachelor's Degree", "Master's Degree", "Some College"],
    default=["High School", "Bachelor's Degree"]
)

# --- CONSULTA ---
query = f"""
SELECT
    c.student_id, c.hours_studied, c.attendance, c.exam_score,
    e.parent_education, e.family_income, e.access_to_resources,
    s.gender
FROM Calificaciones c
JOIN Entorno e ON c.student_id = e.student_id
JOIN Estudiantes s ON c.student_id = s.student_id
WHERE e.parent_education IN {tuple(nivel_educativo) if len(nivel_educativo) > 1 else "('" + nivel_educativo[0] + "')"}
"""
df = pd.read_sql(query, conn)

# --- MÉTRICAS ---
col1, col2, col3 = st.columns(3)
col1.metric("Promedio Examen", f"{df['exam_score'].mean():.2f}")
col2.metric("Horas de Estudio Prom.", f"{df['hours_studied'].mean():.1f}h")
col3.metric("Asistencia Promedio", f"{df['attendance'].mean():.1f}%")

st.divider()

# --- GRÁFICOS ---
fila1_col1, fila1_col2 = st.columns(2)

with fila1_col1:
    st.subheader("Relación Horas de Estudio vs Nota")
    fig1 = px.scatter(df, x="hours_studied", y="exam_score", color="gender", template="plotly_white")
    st.plotly_chart(fig1, use_container_width=True)

with fila1_col2:
    st.subheader("Distribución de Notas por Género")
    fig2 = px.box(df, x="gender", y="exam_score", color="gender")
    st.plotly_chart(fig2, use_container_width=True)

st.subheader("Vista Detallada de la Base de Datos")
st.dataframe(df.head(20), use_container_width=True)

conn.close()

Overwriting app.py


In [17]:
!pkill -f streamlit
!pkill -f localtunnel

In [18]:
!curl ipv4.icanhazip.com

8.228.75.241


In [19]:
# Instalar lo necesario
!pip install streamlit -q
!npm install -g localtunnel -q

# Ejecutar streamlit y abrir el túnel
import subprocess
subprocess.Popen(["streamlit", "run", "app.py"])
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦your url is: https://olive-cups-knock.loca.lt
^C


In [20]:
with open('requirements.txt', 'w') as f:
    f.write('streamlit\npandas\nplotly\n')
print("Archivo requirements.txt listo para descargar.")

Archivo requirements.txt listo para descargar.
